In [1]:
import torch
import numpy as np
import torchvision
import torch.nn as nn
from torchvision.datasets import mnist

In [4]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.Grayscale(num_output_channels=1),
    torchvision.transforms.ToTensor(),
    # Removed normalization to keep data in [0, 1] range for consistency with Noising function and loss
    # torchvision.transforms.Normalize((0.5,), (0.5,))
])

# Load the full MNIST training data
full_train_data = mnist.MNIST(root="./data", train=True, transform=transform, download=True)

# Filter the dataset to include only images with label 7
seven_indices = [i for i, (image, label) in enumerate(full_train_data) if label == 7]
train_data_seven = torch.utils.data.Subset(full_train_data, seven_indices)

# Use the filtered dataset for training
train_data = train_data_seven

In [5]:
def Noising(data_tensor, num_steps, theta = 4):
    for i in range(num_steps):
        data_tensor = theta * data_tensor * (1 - data_tensor)
    noisy_image = data_tensor
    return noisy_image

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Basic double conv block with optional time embedding
# -----------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim=None):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        if hidden_dim is not None:
            self.time_emb_proj = nn.Conv2d(hidden_dim, out_channels, kernel_size=1)
        else:
            self.time_emb_proj = None

    def forward(self, x, t_emb=None):
        x = self.double_conv(x)
        if t_emb is not None and self.time_emb_proj is not None:
            t_emb_mapped = self.time_emb_proj(t_emb)
            x = x + t_emb_mapped
        return x

# -----------------------------
# Downsample block
# -----------------------------
class Down(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim=None):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels, hidden_dim)

    def forward(self, x, t_emb=None):
        x = self.pool(x)
        x = self.conv(x, t_emb)
        return x

# -----------------------------
# Upsample block
# -----------------------------
class Up(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim=None):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_channels, out_channels, hidden_dim)

    def forward(self, x1, x2, t_emb=None):
        x1 = self.up(x1)
        # pad if needed
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        x = self.conv(x, t_emb)
        return x

# -----------------------------
# Output conv
# -----------------------------
class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

# -----------------------------
# Full UNet with t_emb
# -----------------------------
class unet(nn.Module):
    def __init__(self, img_channels=1, hidden_dim=64):
        super().__init__()
        # Encoder
        self.inc = DoubleConv(img_channels, hidden_dim, hidden_dim)
        self.down1 = Down(hidden_dim, hidden_dim*2, hidden_dim)
        self.down2 = Down(hidden_dim*2, hidden_dim*4, hidden_dim)
        self.down3 = Down(hidden_dim*4, hidden_dim*8, hidden_dim)
        self.down4 = Down(hidden_dim*8, hidden_dim*8, hidden_dim)

        # Decoder
        self.up1 = Up(hidden_dim*16, hidden_dim*4, hidden_dim)
        self.up2 = Up(hidden_dim*8, hidden_dim*2, hidden_dim)
        self.up3 = Up(hidden_dim*4, hidden_dim, hidden_dim)
        self.up4 = Up(hidden_dim*2, hidden_dim, hidden_dim)

        # Output
        self.outc = OutConv(hidden_dim, img_channels)

        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, x, t):
        t = t.view(-1, 1).float() / 1000.0
        t_emb = self.time_mlp(t)[:, :, None, None]  # (B, hidden_dim, 1, 1)

        # Encoder
        x1 = self.inc(x, t_emb)
        x2 = self.down1(x1, t_emb)
        x3 = self.down2(x2, t_emb)
        x4 = self.down3(x3, t_emb)
        x5 = self.down4(x4, t_emb)

        # Decoder
        x = self.up1(x5, x4, t_emb)
        x = self.up2(x, x3, t_emb)
        x = self.up3(x, x2, t_emb)
        x = self.up4(x, x1, t_emb)

        out = self.outc(x)
        return out

# -----------------------------
# Test
# -----------------------------
if __name__ == "__main__":
    model = unet(img_channels=1, hidden_dim=64)
    x = torch.randn(2, 1, 64, 64)
    t = torch.tensor([10, 50])
    out = model(x, t)
    print(out.shape)  # (2,1,64,64)

torch.Size([2, 1, 64, 64])


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class deepmodel(nn.Module):
  def __init__(self, shape=torch.tensor([1*28*28]), img_channels=1, hidden_dim=64): # Adjusted default shape
    super().__init__()
    self.input = nn.Linear(shape.prod(), hidden_dim) # Use shape.prod() to get the flattened size
    self.hidden1 = nn.Linear(hidden_dim, hidden_dim)
    self.hidden2 = nn.Linear(hidden_dim, 2*hidden_dim)
    self.hidden3 = nn.Linear(2*hidden_dim, 3*hidden_dim)
    self.hidden4 = nn.Linear(3*hidden_dim, 2*hidden_dim)
    self.hidden5 = nn.Linear(2*hidden_dim, hidden_dim)
    self.output = nn.Linear(hidden_dim, shape.prod()) # Use shape.prod() for output size

  def forward(self, x):
    x = F.relu(self.input(x))
    # x = F.relu(self.hidden1(x))
    x = F.relu(self.hidden2(x))
    x = F.relu(self.hidden3(x))
    x = F.relu(self.hidden4(x))
    x = F.relu(self.hidden5(x))
    x = self.output(x)
    return x

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BetaDeviationLoss(nn.Module):
    def __init__(self, gamma=0.5, normalization_const =1.0, eps=1e-6, reduction='mean', learn_gamma=False):
        """
        A “beta-deviation” style loss that penalizes both being too close and too far.

        Args:
            gamma: initial gamma parameter (>0). If learn_gamma=False, this is fixed.
            R: normalization range to scale error into [0,1].
            eps: small constant to avoid log(0).
            reduction: 'mean' / 'sum' / 'none'
            learn_gamma: if True, gamma becomes a learnable parameter (in log-space).
        """
        super().__init__()
        self.normalization_const = normalization_const
        self.eps = eps
        self.reduction = reduction

        if learn_gamma:
            # store log_gamma as a parameter, so gamma = exp(log_gamma) > 0
            self.log_gamma = nn.Parameter(torch.log(torch.tensor(gamma, dtype=torch.float32)))
        else:
            self.register_buffer('fixed_gamma', torch.tensor(gamma, dtype=torch.float32))
            self.log_gamma = None

    def forward(self, pred, target):
        """
        pred, target: tensors of same shape
        returns: scalar loss (depending on reduction)
        """
        # Determine gamma
        if self.log_gamma is not None:
            gamma = torch.exp(self.log_gamma)
        else:
            gamma = self.fixed_gamma

        # normalized error in (0,1)
        e = torch.abs(pred - target) / self.normalization_const
        e = torch.clamp(e, self.eps, 1.0 - self.eps)

        # simplified loss (drop log B constant if not needed)
        loss_elem = -(gamma - 1.0) * (torch.log(e) + torch.log(1.0 - e))

        # if you want exact negative log-likelihood include log B(gamma,gamma)
        # logB = torch.lgamma(torch.tensor(gamma)) * 2 - torch.lgamma(torch.tensor(2*gamma))
        # loss_elem = loss_elem + logB

        # apply reduction
        if self.reduction == 'mean':
            return loss_elem.mean()
        elif self.reduction == 'sum':
            return loss_elem.sum()
        else:  # 'none'
            return loss_elem

In [9]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import random # Import random

def train(num_epochs, max_noising_steps, model, device="cpu"): # Changed parameter name
    model.train()
    # Adjust normalization_const for loss based on input range [-1, 1]
    # criterion = BetaDeviationLoss(gamma=0.5, normalization_const=2.0, eps=1e-3, learn_gamma=True)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Add a learning rate scheduler
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

    # Use the filtered train_data_seven
    train_loader = DataLoader(train_data_seven, batch_size=128, shuffle=True)

    # Optional: Add print statement to check data range before training loop
    # print("Checking first batch of images:")
    # for images, _ in train_loader:
    #     print("Images shape:", images.shape)
    #     print("Images min:", images.min())
    #     print("Images max:", images.max())
    #     print("Images sample:", images[0, 0, :5, :5]) # Print a small sample of the first image
    #     break # Only check the first batch


    for epoch in range(num_epochs):
        total_loss = 0.0

        for images, _ in train_loader:
            images = images.to(device)

            # Generate random noising steps for each image in the batch
            # Corrected size argument for torch.randint
            t = torch.randint(1, max_noising_steps + 1, (images.size(0),), device=device) # Random steps per image

            noisy_images = torch.stack([Noising(img.unsqueeze(0), step.item(), theta=4).squeeze(0) for img, step in zip(images, t)]).to(device)

            # Normalize noisy images to [-1, 1] before inputting to the model
            # noisy_images_normalized = noisy_images * 2.0 - 1.0 # Map [0, 1] to [-1, 1]

            # The model is expected to output in the same range as the input, so target should also be normalized
            # images_normalized = images * 2.0 - 1.0 # Map [0, 1] to [-1, 1]

            # Flatten the images and noisy images for the deepmodel
            images_flat = images.view(images.size(0), -1)
            noisy_images_flat = noisy_images.view(noisy_images.size(0), -1)


            pred_x_t = model(noisy_images_flat)
            # Calculate loss with normalized target
            loss = criterion(pred_x_t, images_flat)


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        # Step the scheduler after each epoch
        scheduler.step()

        avg_loss = total_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")
        # Optional: Print current learning rate
        # print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")


    torch.save(model.state_dict(), 'diffusion_deepmodel_model.pth')
    print("✅ Model weights saved to diffusion_deepmodel_model.pth")


# ---- Run training ----
model = deepmodel(shape=torch.tensor([1*28*28])) # Changed to deepmodel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train(num_epochs=30, max_noising_steps=30, model=model, device=device) # Increased epochs and max_noising_steps

/var/folders/5f/h7zss1k17c1d8yj6vfm9ds5h0000gn/T/ipykernel_18156/3109888034.py:60: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Scalar.cpp:23.)
  total_loss += loss.item() * images.size(0)


Epoch [1/30], Loss: 0.0549
Epoch [2/30], Loss: 0.0479
Epoch [3/30], Loss: 0.0479
Epoch [4/30], Loss: 0.0479
Epoch [5/30], Loss: 0.0479
Epoch [6/30], Loss: 0.0479


KeyboardInterrupt: 

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Initialize your model
# -----------------------------

a = np.random.uniform(0, 5)
b = np.random.uniform(0, 5)
t = torch.randint(1, 31, (1,))
model = deepmodel(shape=torch.tensor([1*28*28])) # Changed to deepmodel and adjusted shape

# -----------------------------
# Load saved weights
# -----------------------------
checkpoint_path = "diffusion_deepmodel_model_7.pth" # Changed checkpoint path
model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
model.eval()  # set model to evaluation mode

# -----------------------------
# Generate a 28x28 noisy image from Beta(0.5, 0.5)
# -----------------------------
alpha, beta = 0.5, 0.5
print(f"Using Beta parameters: alpha={alpha}, beta={beta}")
print(t)
noisy_image = np.random.beta(alpha, beta, size=(28, 28))
# print(noisy_image)

# Display the noisy image
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(noisy_image, cmap='gray', vmin=0, vmax=1)
plt.title('Noisy (Beta(0.5,0.5))')
plt.axis('off')

# -----------------------------
# Run model inference
# -----------------------------
noisy_tensor = torch.tensor(noisy_image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,28,28)
# Flatten the noisy tensor for the deepmodel
noisy_tensor_flat = noisy_tensor.view(noisy_tensor.size(0), -1)

t_tensor = t  # diffusion / chaotic step # This is not used by deepmodel, but kept for consistency if needed later
with torch.no_grad():  # disable gradients for inference
    reconstructed_tensor_flat = model(noisy_tensor_flat)
    # Reshape the output back to image shape
    reconstructed_tensor = reconstructed_tensor_flat.view(noisy_tensor.size())


reconstructed_image = reconstructed_tensor.squeeze().cpu().numpy()

# Display the reconstructed image
plt.subplot(1, 2, 2)
plt.imshow(reconstructed_image, cmap='gray', vmin=0, vmax=1)
plt.title('Reconstructed')
plt.axis('off')

plt.tight_layout()
plt.show()

NameError: name 'deepmodel' is not defined